In [1]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [20]:
import pandas as pd
import numpy as np
df = pd.read_csv("../data/processed/merged_fraud.csv")
df.head()

,user_id,signup_time,purchase_time,purchase_value,device_id,source,browser,sex,age,ip_address,class,lower_bound_ip_address,upper_bound_ip_address,country
0,247547,2015-06-28 03:00:34,2015-08-09 03:57:29,47,KIXYSVCHIPQBR,SEO,Safari,F,30,1.677886e+07,0,16778240.0,16779263.0,Australia
1,220737,2015-01-28 14:21:11,2015-02-11 20:28:28,15,PKYOWQKWGJNJI,SEO,Chrome,F,34,1.684205e+07,0,16809984.0,16842751.0,Thailand
2,390400,2015-03-19 20:49:09,2015-04-11 23:41:23,44,LVCSXLISZHVUO,Ads,IE,M,29,1.684366e+07,0,16843264.0,16843775.0,China
3,69592,2015-02-24 06:11:57,2015-05-23 16:40:14,55,UHAUHNXXUADJE,Direct,Chrome,F,30,1.693873e+07,0,16924672.0,16941055.0,China
4,174987,2015-07-07 12:58:11,2015-11-03 04:04:30,51,XPGPMOHIDRMGE,SEO,Chrome,F,37,1.697198e+07,0,16941056.0,16973823.0,Thailand


In [8]:
X = df.drop('class', axis=1)
y = df['class']

We separate independent variables (features) from the target variable (fraud label) to prepare the dataset for machine learning.

In [9]:
X = X.drop(['signup_time', 'purchase_time'], axis=1)

Raw timestamp columns are removed because meaningful temporal features (hour, day, time difference) have already been extracted.

In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

Stratified splitting ensures that both training and test sets maintain the same fraud-to-non-fraud ratio, which is critical for imbalanced classification problems.

In [11]:
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train target distribution:\n", y_train.value_counts(normalize=True))

Train shape: (103316, 11)
Test shape: (25830, 11)
Train target distribution:
 class
0    0.90501
1    0.09499
Name: proportion, dtype: float64


We verify dataset split integrity and ensure class imbalance is preserved in both sets.

In [12]:
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

numeric_features, categorical_features

(Index(['user_id', 'purchase_value', 'age', 'ip_address',
        'lower_bound_ip_address', 'upper_bound_ip_address'],
       dtype='object'),
 Index(['device_id', 'source', 'browser', 'sex', 'country'], dtype='object'))

Separating numerical and categorical features allows us to apply appropriate preprocessing techniques such as scaling and encoding.

In [13]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from imblearn.over_sampling import SMOTE

We import tools for feature scaling, encoding categorical variables, and handling class imbalance using SMOTE.

In [14]:
numeric_features = X_train.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X_train.select_dtypes(include=['object']).columns

We identify numerical and categorical columns separately to apply appropriate transformations.

In [15]:
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(handle_unknown='ignore')

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

Numerical features are standardized to ensure equal contribution to the model, while categorical features are encoded into numerical format using one-hot encoding.

In [16]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

The preprocessing pipeline is fitted only on training data to avoid data leakage, then applied to test data.

In [17]:
smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(
    X_train_processed,
    y_train
)

SMOTE is applied only to the training set to synthetically generate minority (fraud) samples. This improves model learning while keeping evaluation realistic.

In [21]:
print("Before SMOTE:", np.bincount(y_train))
print("After SMOTE:", np.bincount(y_train_resampled))

Before SMOTE: [93502  9814]
After SMOTE: [93502 93502]


We verify that SMOTE successfully balanced the training dataset.